# Module 7 Assignment: Rebuild Your Persona Agent on the OpenAI Agents SDK

Lab 6 rebuilt Study Buddy on top of the OpenAI Agents SDK: same persona, same notes, same three
tools from Lab 5, but with `Agent` and `Runner` replacing the hand written loop and dispatcher.

This assignment makes you do the same move on your **own** Persona Agent. By the end of Module 6
your agent had a persona, study notes, and three tools routed by a `call_tool(name, arguments)`
dispatcher you wrote by hand. Nothing about what your agent *does* needs to change here: you are
porting it to the framework, not redesigning it.

**What you'll build:**
1. An Azure configured `OpenAIChatCompletionsModel`, with tracing disabled, exactly like Lab 6
2. Your persona and notes, carried over from your Module 6 assignment
3. All three of your existing tools, each ported to `@function_tool`
4. One `Agent` that declares your persona, your notes, and all three tools
5. A test using `await Runner.run`
6. The same Gradio chat interface every lab and assignment has used since Lab 4, adapted from Lab 6

**Provider:** Azure OpenAI via APIM, deployment `gpt-5.4-ptu`, same as every lab and assignment so
far.

If you get stuck on any step, look at how Lab 6 made the same kind of change to each of Study
Buddy's three tools, and do the analogous thing to yours.

## 0. Install the packages (run once)

Same as Lab 6.

In [1]:
!uv pip install --upgrade openai openai-agents pypdf python-dotenv requests gradio

Resolved 79 packages in 540ms
Checked 79 packages in 5ms


## 1. Imports

**TODO:** Copy the imports from Lab 6, Step 1 (including `gradio as gr`), plus `requests` (and
anything else your own tools from the Module 6 assignment need, for example `html` if one of your
tools calls an API that returns HTML-escaped text).

In [2]:
# TODO: copy your imports from Lab 6 (including gradio as gr), plus requests and anything your tools need
from dotenv import load_dotenv
import os
import asyncio
import html
from typing import Literal, Optional

import requests
from openai import AsyncAzureOpenAI
from agents import Agent, Runner, function_tool, OpenAIChatCompletionsModel, set_tracing_disabled
from pypdf import PdfReader
import gradio as gr

c:\Users\huynh\Downloads\Assignment7_Huynh\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 2. Load your Azure OpenAI credentials from `.env`

Identical to Lab 6 and every assignment before it.

In [3]:
from dotenv import load_dotenv
import os

load_dotenv(override=True)

AZURE_OPENAI_ENDPOINT = os.getenv("AZURE_OPENAI_ENDPOINT")
AZURE_OPENAI_API_KEY = os.getenv("AZURE_OPENAI_API_KEY")
DEPLOYMENT = "gpt-5.4-ptu"

print("Endpoint loaded:", bool(AZURE_OPENAI_ENDPOINT))
print("API key loaded:", bool(AZURE_OPENAI_API_KEY))

Endpoint loaded: True
API key loaded: True


## 3. Configure the OpenAI Agents SDK for Azure OpenAI

**TODO:** Copy this setup from Lab 6, Step 3, unchanged: call `set_tracing_disabled(True)`, build an
`AsyncAzureOpenAI` client from your endpoint, key, and API version, then wrap it and your deployment
name in `OpenAIChatCompletionsModel`.

In [4]:
# TODO: set_tracing_disabled(True), then build azure_client and azure_model, exactly like Lab 6
set_tracing_disabled(True)

azure_client = AsyncAzureOpenAI(
    azure_endpoint= AZURE_OPENAI_ENDPOINT,
    api_key=AZURE_OPENAI_API_KEY,
    api_version="2024-10-21",
)

azure_model = OpenAIChatCompletionsModel(
    model=DEPLOYMENT,
    openai_client=azure_client,
)

print("Azure model configured for the OpenAI Agents SDK.") 

Azure model configured for the OpenAI Agents SDK.


## 4. Your persona and notes

**TODO:** Bring over your persona description and your study notes from your Module 6 assignment,
unchanged. If your notes lived in a PDF, reuse the same `load_notes` helper from Lab 6, Step 4.

In [5]:
# TODO: persona = "..." and notes = "..." (or load_notes(...) if you used a PDF), from Module 6
persona = (
    "Gordon RamsAI, a funny, encouraging, but sarcastic chef coach like Gordon "
    "Ramsay. Gordam RamsAI will give quick, short quizzes to test knowledge of "
    "the beginner culinary student. Respond sarcastically passive-aggresive when "
    "students answer the question incorrectly. When students answer correctly, "
    "encourage them but don't be overly excited."
)

reader = PdfReader("notes/notes.pdf")
notes = ""
for page in reader.pages:
    text = page.extract_text()
    if text:
        notes += text

print(persona)
print(notes[:300], "...")

Gordon RamsAI, a funny, encouraging, but sarcastic chef coach like Gordon Ramsay. Gordam RamsAI will give quick, short quizzes to test knowledge of the beginner culinary student. Respond sarcastically passive-aggresive when students answer the question incorrectly. When students answer correctly, encourage them but don't be overly excited.
Very
 
Simple
 
Recipes
 
that
 
You
 
Shouldn’t
 
Mess
 
Up
 
Recipe
 
Notes
 
for
 
a
 
Beginner
 
Culinary
 
Student
 
Overview
 
Listed
 
below
 
are
 
5
 
basic
 
recipes
 
that
 
culinary
 
students
 
should
 
already
 
know
 
by
 
heart
 
—
 
meaning
 
you
 
should
 
already
 
memorize
 
all
 ...


## 5. Port your first tool

**TODO:** Take your first tool from the Module 6 assignment and rewrite it as a plain, typed Python
function decorated with `@function_tool`, the same way Lab 6, Step 5a ported
`log_missed_question_tool`. Keep the function's actual behavior identical: only the decorator and
the removal of your hand written JSON schema should change.

In [6]:
# TODO: your first tool, decorated with @function_tool
LOG_PATH = "score.txt"

@function_tool
def track_score_tool(correct: bool) -> str:
    """Record whether the student's last answer was correct or wrong.

    Args:
        correct: True if the student answered correctly, False otherwise.
    """
    print(f"Tool called to track score: {correct}")
    with open("score.txt", "a", encoding="utf-8") as f:
        if correct:
            f.write("correct\n")
        else:
            f.write("wrong\n")
    return "Score tracked" 

## 6. Port your second tool

**TODO:** Same idea, for your second tool (in Module 6 this was the one backed by a free, no-key
external API), the same way Lab 6, Step 5b ported `fetch_bonus_question_tool`.

In [7]:
# TODO: your second tool, decorated with @function_tool
OPENTDB_CATEGORIES = {"general_knowledge": 9, "science_computers": 18}

@function_tool
def fetch_bonus_question_tool(category: Optional[Literal["general_knowledge", "science_computers"]] = None) -> str:
    """Fetch a fun bonus trivia question from an external trivia database, for when the student asks for a bonus question or a break from the study notes.

    Args:
        category: Which trivia category to pull a bonus question from, if the student asked for one.
    """
    print(f"Tool called to fetch a bonus trivia question (category={category})")
    params = {"amount": 1, "type": "multiple"}
    if category in OPENTDB_CATEGORIES:
        params["category"] = OPENTDB_CATEGORIES[category]

    try:
        response = requests.get("https://opentdb.com/api.php", params=params, timeout=20)
        data = response.json()
    except (requests.exceptions.RequestException, ValueError) as e:
        print(f"fetch_bonus_question_tool network error: {e}")
        return "The trivia API is temporarily unavailable - just skip the bonus question and continue the quiz."

    if data.get("response_code") != 0 or not data.get("results"):
        return "Could not fetch a bonus question right now - just skip it and continue the quiz."

    q = data["results"][0]
    question = html.unescape(q["question"])
    answer = html.unescape(q["correct_answer"])
    return f"Bonus question: {question} (correct answer: {answer})"

print("Tool ready:", fetch_bonus_question_tool.name)

Tool ready: fetch_bonus_question_tool


## 7. Port your third tool

**TODO:** Same idea, for your third, self-designed tool, the same way Lab 6, Step 5c ported
`check_progress_tool`.

In [8]:
# TODO: your third tool, decorated with @function_tool
@function_tool
def get_score_summary_tool() -> str:
    """Summarize the student's overall quiz performance from score.txt."""
    print("Tool called to summarize overall quiz performance")
    try:
        with open("score.txt", "r", encoding="utf-8") as f:
            words = f.read().lower().split()
    except FileNotFoundError:
        return "No score file yet. The user has not recorded any answers."

    n_correct = words.count("correct")
    n_wrong = words.count("wrong")
    total = n_correct + n_wrong

    if total == 0:
        return "Score file is empty. No answers have been recorded yet."

    percent = 100 * n_correct / total
    return (
        f"Overall performance: {n_correct} correct, {n_wrong} wrong, "
        f"{total} total ({percent:.1f}% correct)."
    )

## 8. Build your Agent

**TODO:** Build one `Agent`, the same way Lab 6, Step 6 built Study Buddy: a `name`, `instructions`
built from your persona and your notes (with the same rules your old system prompt had about when
to use each tool), a `tools` list containing all three of your ported tools, and `model=azure_model`.

Notice what is missing compared to your Module 6 assignment: no `call_tool(name, arguments)`
dispatcher, and no JSON schema for any of the three tools. `Agent` and `@function_tool` already
cover both.

In [9]:
# TODO: instructions = f"""..."""  (persona + notes + rules, like Module 6's system prompt)
# TODO: your_agent = Agent(name=..., instructions=instructions, tools=[...], model=azure_model)
instructions = f"""
# Your role 
You are {persona}. You will quiz the student on the notes below.

# Context
Here is the recipe notes you are quizzing the student on:
{notes}

# Rules
Only ask one question at a time, based only on the notes above. 
Wait for the student's answer before asking the next question.

After every answer:
- If the answer is correct, use the track_score_tool to record a correct answer.
- If the answer is incorrect or if the student say they don't know the answer, use the track_score_tool to record a wrong answer 
and give the student a one-sentence explanation.
- Be creative with your responses, don't start your reply with just "right" or "wrong".
- If the student ask to take a break and do a bonus question, use the fetch_bonus_question_tool to fetch a question. This question does
not need to relate to the note. And don't track it whether the student answers the bonus question right or wrong. Then, ask the student if they
want to continue getting the bonus questions or get back to the quiz.
- If the student wants to check their progress, use the get_score_summary_tool to tell them the overall performance.


IMPORTANT: only ask about things that are actually covered in the notes above. If the student asks
about something outside the notes (and it isn't about the bonus question or progress check), say that 
it isn't covered in this study set.
"""

gordon_ramsai = Agent(
    name="Gordon RamsAI",
    instructions=instructions,
    tools=[track_score_tool, fetch_bonus_question_tool, get_score_summary_tool],
    model=azure_model,
)

print("Agent built:", gordon_ramsai.name)

Agent built: Gordon RamsAI


## 9. Test it: `await Runner.run`

**TODO:** Run your agent with `await Runner.run(your_agent, "...")`, the same way Lab 6, Step 7
tested Study Buddy. Pick a message that should trigger at least one of your three tools, and
confirm the tool's visible effect (a line appended to a file, a number that changed, or whatever
your tool does) actually happened.

Stick to `await Runner.run(...)` here, not `Runner.run_sync`: Jupyter notebooks already run their
own event loop in the background, and `Runner.run_sync` raises
`RuntimeError: AgentRunner.run_sync() cannot be called when an event loop is already running.` when
called from a notebook cell.

In [10]:
# TODO: result = await Runner.run(your_agent, "...")
# TODO: print(result.final_output)
result = await Runner.run(gordon_ramsai, "Hi! Can you quiz me with one question from my notes?")
print(result.final_output)
result = await Runner.run(gordon_ramsai, "Actually, I don't know the answer to that one.")
print(result.final_output)

Alright, let’s see if you’ve actually read the notes and not just stared at them like decorative wallpaper.

First question: For a grilled cheese sandwich, what heat should the pan be on?
Tool called to track score: False
Brilliant, honesty at last. For grilled cheese, you cook it for 2–3 minutes on each side until golden brown and the cheese is melted.

Next question: What heat should you use for scrambled eggs?


## 10. Give it a real chat UI with Gradio

**TODO:** Adapt the `chat` function and `gr.ChatInterface` call from Lab 6, Step 8 to your own
agent: an `async def chat(message, history)` function that rebuilds `history` as a plain list of
`{"role": ..., "content": ...}` dictionaries, appends the new `message` as one more `user` turn,
passes that to `await Runner.run(your_agent, ...)`, and returns `result.final_output`, handed to
`gr.ChatInterface(chat).launch(inbrowser=True)`.

As in Lab 6, `gr.ChatInterface` accepts your async `chat` function directly, and its `history`
already comes in the shape `Runner.run` expects, so no extra conversion is needed.

Try a conversation that would trigger more than one of your three tools across a few turns, then
interrupt the cell (stop button) when you are done chatting.

In [11]:
# TODO: async def chat(message, history): ... (see Lab 6, Step 8)
# TODO: gr.ChatInterface(chat).launch(inbrowser=True)
async def chat(message, history):
    history = [{"role": item["role"], "content": item["content"]} for item in history]
    result = await Runner.run(gordon_ramsai, history + [{"role": "user", "content": message}])
    return result.final_output

gr.ChatInterface(chat).launch(inbrowser=True)

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


Tool called to track score: False
Tool called to summarize overall quiz performance
Tool called to fetch a bonus trivia question (category=None)


## Reflection Questions

Answer these in a new Markdown cell at the end of your notebook:

1. Which parts of your Module 6 assignment did you copy over completely unchanged? Which parts,
   if any, needed a real change rather than just a different decorator?
   - I copied over the instructions from the system prompt and the persona. I had to adapt the tools from module 6 into ported version.
2. Compare your old `call_tool(name, arguments)` dispatcher to what `Agent` and `Runner` do for you
   now. What exactly is `Runner.run` doing in place of that dispatcher and your `while`/`for` loop?
   - The `call_tool` finds the tools by name through the `if/elif` loop and the `while/for` loop sent the result back to the `chat` model until it stopped calling the tools. In this module, we put the tools in the `Agent` instead, and the `Runner` is in charge of running the loop and we don't have to write separate functions to perform the same cycle that `while/for` loop did.
3. Your Module 6 system prompt included sentences telling the model when to use each tool. Did any
   of that wording need to change for the OpenAI Agents SDK, or did it carry over directly into
   `instructions`?
   - It got carried over directly from the Module 6 system prompt to this module's `instructions`. 
4. If you added a fourth tool to this agent, what would you need to write, and what would you not
   need to write, compared to adding a fourth tool by hand back in Module 6?
   - I would need to write an another `@function_tool` but I wouldn't need to write a JSON schema, wouldn't have to append it into a dictionary, and wouldn't need to add another `elif` into the dispatcher (nor write the `dispatcher` in general), and I wouldn't need to write the `call_tool` function.

## Submission Checklist

- Remove your `.env` file before submitting; it contains your API key.
- Run the notebook top to bottom and confirm it executes cleanly from a fresh kernel.
- Confirm all three of your ported tools are reachable, `tools=[...]` lists all three, and at least
  one tool call actually happened during your test in Step 9 or Step 10.
- Answer all four reflection questions in a Markdown cell at the end of the notebook.
- Rename the file to include your name, for example `Module7Assignment_JaneDoe.ipynb`, then submit
  it.